# Fongbe Dataset Preparation

Download and prepare unified Fongbe ASR dataset on Google Drive.

**Run once** - Takes ~30min

## 1. Setup

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/fongbe')
PROJECT_ROOT.mkdir(exist_ok=True)
print(f"✓ Project: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Project: /content/drive/MyDrive/fongbe


## 2. Install Dependencies

In [2]:
!pip install -q datasets soundfile librosa
print("✓ Dependencies installed")

✓ Dependencies installed


## 3. Download Datasets from HuggingFace

In [3]:
from datasets import load_dataset

print("Downloading Laleye dataset...")
laleye = load_dataset("Laleye/fongbe_asr", split="train", trust_remote_code=True)
print(f"✓ Laleye: {len(laleye)} samples")

print("\nDownloading pyFongbe dataset...")
pyfongbe = load_dataset("Laleye/fongbe_asr", split="validation", trust_remote_code=True)
print(f"✓ pyFongbe: {len(pyfongbe)} samples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/533 [00:00<?, ?B/s]

DataFilesNotFoundError: No (supported) data files found in godwinh/fongbe-asr

## 4. Unify and Split Dataset

In [ ]:
from datasets import Dataset, DatasetDict, Audio

# Unify schema
def process_laleye(batch):
    return {
        "audio": batch["audio"],
        "text": batch["sentence"],
        "source": "laleye"
    }

def process_pyfongbe(batch):
    return {
        "audio": batch["audio"],
        "text": batch["transcription"],
        "source": "pyfongbe"
    }

laleye_clean = laleye.map(process_laleye, remove_columns=laleye.column_names)
pyfongbe_clean = pyfongbe.map(process_pyfongbe, remove_columns=pyfongbe.column_names)

# Concatenate
from datasets import concatenate_datasets
unified = concatenate_datasets([laleye_clean, pyfongbe_clean])

print(f"\n✓ Unified dataset: {len(unified)} samples")

# Split 80/10/10
split = unified.train_test_split(test_size=0.2, seed=42)
test_val = split['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train': split['train'],
    'validation': test_val['train'],
    'test': test_val['test']
})

print(f"\n✓ Train: {len(dataset_dict['train'])}")
print(f"✓ Validation: {len(dataset_dict['validation'])}")
print(f"✓ Test: {len(dataset_dict['test'])}")

## 5. Save to Drive

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "fongbe_asr_unified"

print(f"Saving to {OUTPUT_PATH}...")
dataset_dict.save_to_disk(str(OUTPUT_PATH))

print("\n✓ Dataset saved to Drive!")
print(f"Location: {OUTPUT_PATH}")

## 6. Verify

In [ ]:
from datasets import load_from_disk

# Reload to verify
loaded = load_from_disk(str(OUTPUT_PATH))

print("✓ Verification successful")
print(f"\nTrain: {len(loaded['train'])} samples")
print(f"Columns: {loaded['train'].column_names}")
print(f"\nExample: {loaded['train'][0]['text'][:100]}...")